# 04 — Train-only feature inspection and optional ARIMA
**Independent 2026 research demonstration.** Correlated features are not automatically predictive; tree impurity importance can favor persistent price-level signals. Feature selection must not use test data, and ARIMA must be evaluated on chronological future observations.

In [ ]:
from pathlib import Path
from sp500_research import load_prices, download_sp500, build_supervised_frame, chronological_split
DATA_FILE = Path('../data/sp500_ohlcv.csv')
if DATA_FILE.exists():
    prices = load_prices(DATA_FILE)
else:
    # Explicit opt-in: new Yahoo Finance download, not the original 2024 data vintage.
    prices = download_sp500(start='2000-01-01', end='2025-01-01')
print('Data coverage:', prices.index.min().date(), 'to', prices.index.max().date(), len(prices), 'daily rows')

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sp500_research.features import model_features
frame=build_supervised_frame(prices);p=chronological_split(frame)
cols=model_features(frame)
rf=RandomForestRegressor(n_estimators=100,min_samples_leaf=4,random_state=42,n_jobs=-1)
rf.fit(p.train[cols],p.train.target_next_close)
importance=permutation_importance(rf,p.valid[cols],p.valid.target_next_close,
                                   scoring='neg_mean_absolute_error',n_repeats=3,
                                   random_state=42,n_jobs=1)
ranks=pd.Series(importance.importances_mean,index=cols).sort_values(ascending=False)
print('Validation permutation importance (MAE units):')
print(ranks.head(12).round(4).to_string())

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(8,4))
ranks.head(10).sort_values().plot.barh(ax=ax)
ax.set(title='Validation-period permutation importance',xlabel='Change in neg-MAE')
plt.tight_layout();plt.show()

In [ ]:
# Optional ARIMA illustration: rolling one-step forecast.
# Use end-of-day close through t when predicting t+1.  Walk only a bounded
# test slice to keep a notebook demo manageable.
try:
    from statsmodels.tsa.arima.model import ARIMA
except ImportError as exc:
    raise ImportError('Install statsmodels to run optional ARIMA comparison') from exc
from sp500_research.evaluation import regression_metrics
point_dates=p.test.index[:40]
raw_close=prices.Close
seed_dates=raw_close.index[raw_close.index<=point_dates[0]]
res=ARIMA(raw_close.loc[seed_dates],order=(1,1,0)).fit()
forecasts=[]
for i,date in enumerate(point_dates):
    # Always ingest only *currently observed* close, never the future target.
    if raw_close.index[-1] < date:
        raise ValueError('Insufficient market history')
    if i>0:
        res=res.append(raw_close.loc[[date]],refit=False)
    forecasts.append(float(res.forecast(steps=1).iloc[0]))
actual=p.test.iloc[:len(forecasts)].target_next_close.to_numpy()
print('ARIMA next-day demo:',regression_metrics(actual,forecasts))
print('Naive close:',regression_metrics(actual,p.test.iloc[:len(forecasts)].close_t))

### Interpretation
Importance measurements use validation features and labels. ARIMA is **not** fitted on the published-paper dataset and is **not** directly comparable to the 2024 saved values. No claim of temporal causality or stable investment performance is made.